# ISTAT parser walkthrough

This notebook is the practical guide for parsing official ISTAT input-output releases in MARIO.


## What this notebook covers

- where the official ISTAT releases come from;
- when to pass one workbook, one extracted release directory, or one zip archive;
- how `IOT` and `SUT` workflows differ;
- how `iot_mode=`, `level=`, `price=`, and `valuation=` are used;
- how `download=True` works with the official ISTAT release pages;
- which caveats matter for this parser.


## Relevant source page

- Official ISTAT tag page: [ISTAT sistema input-output](https://www.istat.it/tag/sistema-input-output/)

This parser targets the official ISTAT release bundles, not arbitrary spreadsheets with a similar shape.


## Main entry point

For normal user workflows, the public entry point is:

- `mario.parse_istat(...)`

The same function supports:

- `IOT` parsing;
- `SUT` parsing;
- local parsing from one workbook, one extracted release directory, or one release zip;
- automatic download of the official release through `download=True`.


## Key arguments

The key public arguments are:

- `path`: local workbook, extracted release directory, release zip, or cache directory;
- `year`: reference year to select from the ISTAT release;
- `table`: choose `"IOT"` or `"SUT"`;
- `iot_mode`: for `IOT`, either `"product"` or `"industry"`;
- `level`: for `SUT`, either `"63"` or `"20"`;
- `price`: for `SUT`, either `"current"` or `"pyp"`;
- `valuation`: for `SUT`, either `"basic"` or `"purchaser"`;
- `download`: when `True`, MARIO downloads the official release before parsing it;
- `edition` and `page_url`: downloader selectors for the official ISTAT release page.


## `IOT` versus `SUT`

Use `table="IOT"` when you want the symmetric representation with `Z`, `Y`, and `V`.

Use `table="SUT"` when you want the native split structure with `S`, `U`, `Yc`, `Ya`, `Va`, and `Vc`.

For `IOT`, the key selector is `iot_mode=`. For `SUT`, the key selectors are `level=`, `price=`, and `valuation=`.


## Workbook, extracted directory, or zip archive

For `IOT`, passing the workbook directly is often the simplest workflow.

For `SUT`, passing the extracted release directory or the release zip is usually more convenient, because MARIO resolves the required workbook triplet from the official file names.


## Download workflow

If you already have the official release locally, point the parser to the workbook, directory, or zip file.

If you want MARIO to fetch the official release first, use `download=True` together with a cache directory in `path`.


In [ ]:
import mario

## Parse one local `IOT` workbook

This is the simplest `IOT` workflow.


In [ ]:
db = mario.parse_istat(
    path="/path/to/istat_release",
    year=2022,
    table="IOT",
    iot_mode="product",
    calc_all=False,
)

db

## Switch the `IOT` mode

Use `iot_mode="industry"` when you want the industry-by-industry release instead of the product-by-product one.


In [ ]:
db = mario.parse_istat(
    path="/path/to/istat_release",
    year=2022,
    table="IOT",
    iot_mode="industry",
    calc_all=False,
)

db

## Parse one `SUT` release

The `SUT` workflow needs the structural selectors explicitly.


In [ ]:
db = mario.parse_istat(
    path="/path/to/istat_release",
    year=2022,
    table="SUT",
    level="63",
    price="current",
    valuation="basic",
    calc_all=False,
)

db

## Parse from a release zip

You can point the parser directly to the official ISTAT release zip instead of extracting it yourself.


In [ ]:
db = mario.parse_istat(
    path="/path/to/istat_release.zip",
    year=2022,
    table="SUT",
    level="20",
    price="pyp",
    valuation="purchaser",
    calc_all=False,
)

db

## Download and parse in one step

This is useful when you want a reproducible local cache of the official release.


In [ ]:
db = mario.parse_istat(
    path="/path/to/istat_cache",
    year=2022,
    table="IOT",
    iot_mode="product",
    download=True,
    edition="latest",
    calc_all=False,
)

db

## First inspection

Once the release is parsed, the normal MARIO inspection helpers work exactly as for the other parsers.


In [ ]:
db.table_type
db.sets
db.units

## Caveats

- the parser expects the official ISTAT workbook and release naming conventions;
- `SUT` parsing has more structural selectors than `IOT`, so explicit arguments are usually the better choice;
- `download=True` is optional, but it is often the cleanest workflow when you want a stable local archive of the release.
